In [19]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, VotingRegressor, BaggingRegressor
from sklearn.neighbors import KNeighborsRegressor

In [20]:
df = pd.read_csv('Life Expectancy Data.csv')
print("Dataset Shape:", df.shape)
display(df.head())

Dataset Shape: (2938, 22)


,Country,Year,Status,Life expectancy,Adult Mortality,infant deaths,Alcohol,percentage expenditure,Hepatitis B,Measles,...,Polio,Total expenditure,Diphtheria,HIV/AIDS,GDP,Population,thinness 1-19 years,thinness 5-9 years,Income composition of resources,Schooling
0,Afghanistan,2015,Developing,65.0,263.0,62,0.01,71.279624,65.0,1154,...,6.0,8.16,65.0,0.1,584.259210,33736494.0,17.2,17.3,0.479,10.1
1,Afghanistan,2014,Developing,59.9,271.0,64,0.01,73.523582,62.0,492,...,58.0,8.18,62.0,0.1,612.696514,327582.0,17.5,17.5,0.476,10.0
2,Afghanistan,2013,Developing,59.9,268.0,66,0.01,73.219243,64.0,430,...,62.0,8.13,64.0,0.1,631.744976,31731688.0,17.7,17.7,0.470,9.9
3,Afghanistan,2012,Developing,59.5,272.0,69,0.01,78.184215,67.0,2787,...,67.0,8.52,67.0,0.1,669.959000,3696958.0,17.9,18.0,0.463,9.8
4,Afghanistan,2011,Developing,59.2,275.0,71,0.01,7.097109,68.0,3013,...,68.0,7.87,68.0,0.1,63.537231,2978599.0,18.2,18.2,0.454,9.5


In [21]:
df.columns = df.columns.str.strip()

In [22]:
print("Missing Values before imputation:\n", df.isnull().sum())

Missing Values before imputation:
 Country                              0
Year                                 0
Status                               0
Life expectancy                     10
Adult Mortality                     10
infant deaths                        0
Alcohol                            194
percentage expenditure               0
Hepatitis B                        553
Measles                              0
BMI                                 34
under-five deaths                    0
Polio                               19
Total expenditure                  226
Diphtheria                          19
HIV/AIDS                             0
GDP                                448
Population                         652
thinness  1-19 years                34
thinness 5-9 years                  34
Income composition of resources    167
Schooling                          163
dtype: int64


In [23]:
for col in df.select_dtypes(include=[np.number]).columns:
    df[col].fillna(df[col].median(), inplace=True)

/var/folders/70/s6bttn7d6_qdr8b28nj_vt640000gn/T/ipykernel_62213/3880857199.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df[col].fillna(df[col].median(), inplace=True)
/var/folders/70/s6bttn7d6_qdr8b28nj_vt640000gn/T/ipykernel_62213/3880857199.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Serie

In [24]:
for col in df.select_dtypes(include=['object']).columns:
    df[col].fillna(df[col].mode()[0], inplace=True)

print("\nMissing Values after imputation:", df.isnull().sum().sum())


Missing Values after imputation: 2563


/var/folders/70/s6bttn7d6_qdr8b28nj_vt640000gn/T/ipykernel_62213/3202541515.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:
/var/folders/70/s6bttn7d6_qdr8b28nj_vt640000gn/T/ipykernel_62213/3202541515.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-

In [25]:
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

/var/folders/70/s6bttn7d6_qdr8b28nj_vt640000gn/T/ipykernel_62213/3790181639.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns


In [30]:
from sklearn.impute import SimpleImputer

df = df.dropna(subset=['Life expectancy'])

In [31]:
X = df.drop('Life expectancy', axis=1)
y = df['Life expectancy']

In [32]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [33]:
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

In [34]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(random_state=42),
    "Lasso Regression": Lasso(random_state=42),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=42),
    "Random Forest Regressor": RandomForestRegressor(random_state=42),
    "Gradient Boosting Regressor": GradientBoostingRegressor(random_state=42),
    "AdaBoost Regressor": AdaBoostRegressor(random_state=42),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Bagging Regressor": BaggingRegressor(random_state=42)
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        "Model": name,
        "R2 Score": r2,
        "RMSE": rmse,
        "MAE": mae,
        "MSE": mse
    })
    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values(by="R2 Score", ascending=False)

In [35]:
print("\n==================================================")
print("   MODEL PERFORMANCE COMPARISON TABLE  ")
print("==================================================")
display(results_df)

# Select best model based on highest R2 Score
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
print(f"\n Best Performing Model: {best_model_name}")


   MODEL PERFORMANCE COMPARISON TABLE  


,Model,R2 Score,RMSE,MAE,MSE
4,Random Forest Regressor,0.968165,1.659441,1.031544,2.753743
8,Bagging Regressor,0.960480,1.848902,1.151553,3.418438
5,Gradient Boosting Regressor,0.948466,2.111309,1.505403,4.457625
3,Decision Tree Regressor,0.918050,2.662455,1.514334,7.088669
7,K-Neighbors Regressor,0.905760,2.855119,1.862321,8.151706
6,AdaBoost Regressor,0.905603,2.857500,2.267413,8.165306
0,Linear Regression,0.820812,3.936958,2.928892,15.499638
1,Ridge Regression,0.820558,3.939748,2.927551,15.521612
2,Lasso Regression,0.788040,4.281869,3.138290,18.334405



 Best Performing Model: Random Forest Regressor


In [36]:
joblib.dump((best_model, scaler), 'life_expectancy_model.pkl')
print("Model and scaler successfully saved as 'life_expectancy_model.pkl'")

Model and scaler successfully saved as 'life_expectancy_model.pkl'
